# 4.1.2 Temperature Sensitivity — Paired t-test by Tier

For each tier, pool all trials across the 16 models (matched by `(model, run_id)`) and run a paired t-test on ISR between $T{=}0.0$ and $T{=}0.7$, no-safety condition.

**What this shows:** whether raising temperature produces a detectable average shift in ISR *at a given tier*, once we control for trial-level pairing (same model, same job, same seed, same K). Tiers with a significant positive shift indicate the tier is sensitive to sampling noise; tiers with no significant shift are temperature-robust.

**Output:** LaTeX table written to `latex/tables/temp_sensitivity.tex`, ready to `\input{}` from the paper.

In [1]:
import sys, json
import os
from pathlib import Path
import numpy as np
from scipy import stats

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
RESULTS = ROOT / 'results' / 'interview_v4'
TABLES_DIR = ROOT / 'outputs' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from analysis.labels import tier_subset, MODELS, MODEL_SLUGS, SLUG_TO_NAME
from analysis.validation import validate_all_json_file

TIERS = tier_subset(include_title_only=True)  # all 7

In [2]:
PERSONA = 'B'  # change to run the analysis on a different persona

def load_trials(slug, cond, temp, persona=PERSONA):
    """Return {run_id: isr} for all trials in a (model, condition, temp, persona) cell."""
    p = RESULTS / cond / slug / f'temp_{temp}' / f'persona{persona}' / 'all_results.json'
    if not p.exists():
        return {}
    trials = validate_all_json_file(
        p, temperature=temp, persona=persona, canonical_models=MODEL_SLUGS
    )
    return {t['run_id']: t['isr'] for t in trials}

def sig_marker(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

# For each tier, pool matched (t=0.0, t=0.7) ISR pairs across all models+run_ids
rows = []
for cond, label in TIERS:
    a_all, b_all = [], []
    n_models_present = 0
    for slug in MODEL_SLUGS:
        t0 = load_trials(slug, cond, 0.0)
        t7 = load_trials(slug, cond, 0.7)
        common = sorted(set(t0) & set(t7))
        if not common:
            continue
        n_models_present += 1
        a_all.extend(t0[k] for k in common)
        b_all.extend(t7[k] for k in common)
    a = np.array(a_all)
    b = np.array(b_all)
    mean0 = a.mean() * 100
    mean7 = b.mean() * 100
    delta = mean7 - mean0
    t_stat, p_val = stats.ttest_rel(b, a)
    rows.append({
        'tier':   label,
        'n':      len(a),
        'n_models': n_models_present,
        'mean_t0':  mean0,
        'mean_t7':  mean7,
        'delta':    delta,
        't':        t_stat,
        'p':        p_val,
        'sig':      sig_marker(p_val),
    })

print(f'Pooled paired t-test (persona {PERSONA}, matched by model × run_id, per tier):')
print()
for r in rows:
    print(f"{r['tier']:<13} n={r['n']:>5}  t0={r['mean_t0']:>5.2f}%  t7={r['mean_t7']:>5.2f}%  "
          f"Δ={r['delta']:+5.2f}%  t={r['t']:>6.2f}  p={r['p']:.3g}  {r['sig']}")

Pooled paired t-test (persona B, matched by model × run_id, per tier):

Baseline      n= 1800  t0= 0.89%  t7= 1.04%  Δ=+0.15%  t=  1.65  p=0.0988  
Job duties    n= 1800  t0= 0.83%  t7= 1.39%  Δ=+0.55%  t=  6.13  p=1.06e-09  ***
Job context   n= 1800  t0= 1.06%  t7= 1.46%  Δ=+0.40%  t=  4.88  p=1.18e-06  ***
Memo          n= 1800  t0= 7.20%  t7= 7.10%  Δ=-0.10%  t= -0.31  p=0.759  
Authority     n= 1800  t0= 2.72%  t7= 3.13%  Δ=+0.40%  t=  2.47  p=0.0136  *
Policy        n= 1800  t0=14.24%  t7=14.06%  Δ=-0.18%  t= -0.53  p=0.596  
Insurance     n= 1800  t0=14.30%  t7=14.21%  Δ=-0.08%  t= -0.25  p=0.806  


In [3]:
def fmt_p(p):
    """Format a p-value for LaTeX math mode."""
    if p < 1e-4:
        exp = int(np.floor(np.log10(p)))
        mantissa = p / (10 ** exp)
        return f'${mantissa:.1f}{{\\times}}10^{{{exp}}}$'
    if p < 0.001:
        return f'${p:.4f}$'
    return f'${p:.3f}$'

# ------ Pooled tier-level LaTeX table ------
caption_pooled = (
    'Paired $t$-tests of ISR at $T{=}0.0$ vs.\\ $T{=}0.7$, per tier, '
    f'pooling matched trials across all {len(MODELS)} models '
    '(no-safety condition). $\\Delta$ is the mean ISR increase from '
    'raising temperature. Significance markers: $*$ $p<0.05$, '
    '$**$ $p<0.01$, $***$ $p<0.001$.'
)

lines = []
lines.append('\\begin{table}[t]')
lines.append('    \\centering')
lines.append('    \\small')
lines.append(f'    \\caption{{{caption_pooled}}}')
lines.append('    \\label{tab:temp-sensitivity}')
lines.append('    \\begin{tabular}{lrrrrrrl}')
lines.append('        \\toprule')
lines.append('        \\textbf{Tier} & \\textbf{$n$} & \\textbf{ISR $T{=}0.0$} & \\textbf{ISR $T{=}0.7$} & \\textbf{$\\Delta$} & \\textbf{$t$} & \\textbf{$p$} & \\textbf{Sig.} \\\\')
lines.append('        \\midrule')
for r in rows:
    lines.append(
        f"        {r['tier']} & {r['n']} & "
        f"{r['mean_t0']:.2f}\\% & {r['mean_t7']:.2f}\\% & "
        f"{r['delta']:+.2f}\\% & {r['t']:.2f} & {fmt_p(r['p'])} & {r['sig']} \\\\"
    )
lines.append('        \\bottomrule')
lines.append('    \\end{tabular}')
lines.append('\\end{table}')

tex_pooled = '\n'.join(lines) + '\n'
out_path = TABLES_DIR / 'temp_sensitivity.tex'
out_path.write_text(tex_pooled)
print(f'Wrote {out_path}')
print()
print(tex_pooled)

Wrote ./latex/tables/temp_sensitivity.tex

\begin{table}[t]
    \centering
    \small
    \caption{Paired $t$-tests of ISR at $T{=}0.0$ vs.\ $T{=}0.7$, per tier, pooling matched trials across all 18 models (no-safety condition). $\Delta$ is the mean ISR increase from raising temperature. Significance markers: $*$ $p<0.05$, $**$ $p<0.01$, $***$ $p<0.001$.}
    \label{tab:temp-sensitivity}
    \begin{tabular}{lrrrrrrl}
        \toprule
        \textbf{Tier} & \textbf{$n$} & \textbf{ISR $T{=}0.0$} & \textbf{ISR $T{=}0.7$} & \textbf{$\Delta$} & \textbf{$t$} & \textbf{$p$} & \textbf{Sig.} \\
        \midrule
        Baseline & 1800 & 0.89\% & 1.04\% & +0.15\% & 1.65 & $0.099$ &  \\
        Job duties & 1800 & 0.83\% & 1.39\% & +0.55\% & 6.13 & $1.1{\times}10^{-9}$ & *** \\
        Job context & 1800 & 1.06\% & 1.46\% & +0.40\% & 4.88 & $1.2{\times}10^{-6}$ & *** \\
        Memo & 1800 & 7.20\% & 7.10\% & -0.10\% & -0.31 & $0.759$ &  \\
        Authority & 1800 & 2.72\% & 3.13\% & +0.40\% & 

## Per-model breakdown

Two analyses, same pairing unit (trial) but different aggregation:

1. **Per-model tier-level $p$-values** — for each `(model, tier)` cell, run a paired $t$-test on the 100 matched trials. Reproduces the earlier `temp_significance.py` output: shows *which* models are temperature-sensitive *at which* tiers.
2. **Model-level cross-tier robustness check** — collapse each `(model, tier)` cell to a single mean, then run one paired $t$-test across the 16 models *per tier* (n=16). This is the independence-respecting confirmation. If it agrees with the pooled result in the first table, the effect is bulletproof.

Analysis 1 generates a second LaTeX table written to `latex/tables/temp_sensitivity_per_model.tex`. Analysis 2 is printed for the paper text only.

In [4]:
# ------ Per-model × per-tier paired t-tests ------
# One t-test per (model, tier) cell on the ~100 matched trial pairs.
# Reports delta + significance marker. NaN-safe for models with
# zero variance (e.g. ISR=0 at both temps across every trial).

per_model = {}  # {slug: {tier_label: {'delta', 'p', 'sig', 'mean_t0', 'mean_t7'}}}
for slug, name, _fam in MODELS:
    per_model[slug] = {}
    for cond, label in TIERS:
        t0 = load_trials(slug, cond, 0.0)
        t7 = load_trials(slug, cond, 0.7)
        common = sorted(set(t0) & set(t7))
        if not common:
            per_model[slug][label] = None
            continue
        a = np.array([t0[k] for k in common])
        b = np.array([t7[k] for k in common])
        if np.allclose(a, b):
            t_stat, p_val = 0.0, 1.0
        else:
            t_stat, p_val = stats.ttest_rel(b, a)
            if np.isnan(p_val):  # fully zero variance safety
                p_val = 1.0
        per_model[slug][label] = {
            'mean_t0': float(a.mean()) * 100,
            'mean_t7': float(b.mean()) * 100,
            'delta':   float((b - a).mean()) * 100,
            't':       float(t_stat),
            'p':       float(p_val),
            'sig':     sig_marker(p_val),
        }

# Console preview: rows = models, cols = tier deltas (with markers)
tier_labels = [label for _, label in TIERS]
hdr = ' | '.join(f'{l[:7]:>7}' for l in tier_labels)
print(f"{'Model':<17} | {hdr}")
print('-' * (20 + 10 * len(tier_labels)))
for slug, name, _ in MODELS:
    cells = []
    for label in tier_labels:
        r = per_model[slug][label]
        if r is None:
            cells.append(f"{'—':>7}")
        else:
            cells.append(f"{r['delta']:+5.1f}{r['sig']:<2}")
    print(f"{name:<17} | " + ' | '.join(f'{c:>7}' for c in cells))

Model             | Baselin | Job dut | Job con |    Memo | Authori |  Policy | Insuran
------------------------------------------------------------------------------------------
GPT-4o-mini       |  +1.1   |  +3.1*** |  +2.9*** |  +3.9*** |  +1.6   |  -0.0   |  -1.8  
GPT-4.1-mini      |  +0.6*  |  +0.8** |  +0.7*  |  +0.2   |  +1.5** |  -1.4   |  -0.2  
GPT-5.4-mini      |  -0.0   |  +0.1   |  +0.2*  |  +0.1   |  +0.0   |  +1.4   |  +1.7  
GPT-5.4           |  +0.0   |  -0.0   |  +0.0   |  +0.5   |  -0.0   |  +1.1   |  +0.4  
Haiku 3.5         |  -0.0   |  +0.1   |  -0.0   |  +0.0   |  -0.0   |  +0.4*  |  +0.1  
Haiku 4.5         |  +0.1   |  +0.0   |  -0.0   |  +0.1   |  +0.1   |  -1.2   |  -0.5  
Sonnet 4.6        |  +0.0   |  +0.1   |  +0.1   |  -1.4   |  -0.0   |  -0.2   |  +0.0  
Opus 4.7          |  +0.0   |  +0.0   |  +0.0   |  -0.1   |  +0.0   |  +0.3   |  +0.2  
Gemini 2.5 Flash  |  +0.0   |  -0.0   |  +0.0   |  +0.2   |  +0.1   |  +0.7   |  -0.1  
Gemini 2.5 Pro    |  +0.0 

In [5]:
# ------ Per-model LaTeX table ------
# Two subcolumns per tier: right-aligned number, then left-aligned sig marker.
# @{} joins them visually so numbers line up cleanly without the superscript
# pushing them off-center.

def fmt_cell_pair(r):
    """Return (number_cell, sig_cell) for the r@{}l column pair."""
    if r is None:
        return ('\\multicolumn{2}{c}{---}', None)  # spans both subcolumns
    num = f"${r['delta']:+.1f}$"
    sig = f"$^{{{r['sig']}}}$" if r['sig'] else ''
    return (num, sig)

caption_per_model = (
    'Per-model paired $t$-tests of ISR at $T{=}0.0$ vs.\\ $T{=}0.7$, '
    'by tier. Each cell is $\\Delta$ ISR (percentage points); '
    'significance markers ($*$ $p<0.05$, $**$ $p<0.01$, $***$ $p<0.001$) '
    'come from a paired $t$-test on $\\sim\\!100$ matched trials. Positive '
    'values indicate higher ISR at $T{=}0.7$. No-safety condition.'
)

# Column spec: model name (l) + per-tier (r@{}l) so numbers align
# and sig markers sit flush against them.
col_spec = 'l' + 'r@{\\,}l' * len(tier_labels)

lines = []
lines.append('\\begin{table}[t]')
lines.append('    \\centering')
lines.append('    \\scriptsize')
lines.append(f'    \\caption{{{caption_per_model}}}')
lines.append('    \\label{tab:temp-sensitivity-per-model}')
lines.append(f'    \\begin{{tabular}}{{{col_spec}}}')
lines.append('        \\toprule')
# Header uses multicolumn to span each (number, sig) pair
header_cells = ['\\textbf{Model}'] + [f'\\multicolumn{{2}}{{c}}{{\\textbf{{{lbl}}}}}' for lbl in tier_labels]
lines.append('        ' + ' & '.join(header_cells) + ' \\\\')
lines.append('        \\midrule')
for slug, name, _ in MODELS:
    row_cells = [name]
    for label in tier_labels:
        num, sig = fmt_cell_pair(per_model[slug][label])
        if sig is None:
            # --- case: single spanning cell
            row_cells.append(num)
        else:
            row_cells.append(num)
            row_cells.append(sig)
    lines.append('        ' + ' & '.join(row_cells) + ' \\\\')
lines.append('        \\bottomrule')
lines.append('    \\end{tabular}')
lines.append('\\end{table}')

tex_per_model = '\n'.join(lines) + '\n'
out_path = TABLES_DIR / 'temp_sensitivity_per_model.tex'
out_path.write_text(tex_per_model)
print(f'Wrote {out_path}')
print()
print(tex_per_model)

Wrote ./latex/tables/temp_sensitivity_per_model.tex

\begin{table}[t]
    \centering
    \scriptsize
    \caption{Per-model paired $t$-tests of ISR at $T{=}0.0$ vs.\ $T{=}0.7$, by tier. Each cell is $\Delta$ ISR (percentage points); significance markers ($*$ $p<0.05$, $**$ $p<0.01$, $***$ $p<0.001$) come from a paired $t$-test on $\sim\!100$ matched trials. Positive values indicate higher ISR at $T{=}0.7$. No-safety condition.}
    \label{tab:temp-sensitivity-per-model}
    \begin{tabular}{lr@{\,}lr@{\,}lr@{\,}lr@{\,}lr@{\,}lr@{\,}lr@{\,}l}
        \toprule
        \textbf{Model} & \multicolumn{2}{c}{\textbf{Baseline}} & \multicolumn{2}{c}{\textbf{Job duties}} & \multicolumn{2}{c}{\textbf{Job context}} & \multicolumn{2}{c}{\textbf{Memo}} & \multicolumn{2}{c}{\textbf{Authority}} & \multicolumn{2}{c}{\textbf{Policy}} & \multicolumn{2}{c}{\textbf{Insurance}} \\
        \midrule
        GPT-4o-mini & $+1.1$ &  & $+3.1$ & $^{***}$ & $+2.9$ & $^{***}$ & $+3.9$ & $^{***}$ & $+1.6$ &  & $-0.0$

In [6]:
# ------ Cross-tier model-level robustness check (for paper text) ------
# For each tier, pair across the 16 models (n=16). Each pairing unit is a
# full model mean (already computed in per_model). Compare with the pooled
# result in the table; if both are significant, the effect is bulletproof.

model_level_rows = []
for cond, label in TIERS:
    a = np.array([per_model[slug][label]['mean_t0'] / 100 for slug, _, _ in MODELS
                  if per_model[slug][label] is not None])
    b = np.array([per_model[slug][label]['mean_t7'] / 100 for slug, _, _ in MODELS
                  if per_model[slug][label] is not None])
    if np.allclose(a, b):
        t_stat, p_val = 0.0, 1.0
    else:
        t_stat, p_val = stats.ttest_rel(b, a)
        if np.isnan(p_val):
            p_val = 1.0
    model_level_rows.append({
        'tier':    label,
        'n':       len(a),
        'mean_t0': float(a.mean()) * 100,
        'mean_t7': float(b.mean()) * 100,
        'delta':   float((b - a).mean()) * 100,
        't':       float(t_stat),
        'p':       float(p_val),
        'sig':     sig_marker(p_val),
    })

print('Model-level paired t-test (n=16 models per tier):')
print()
print(f"{'Tier':<13} {'t=0.0':>7} {'t=0.7':>7} {'Δ':>8} {'t':>7} {'p':>10} {'sig':>4}")
print('-' * 58)
for r in model_level_rows:
    print(f"{r['tier']:<13} {r['mean_t0']:>6.2f}% {r['mean_t7']:>6.2f}% "
          f"{r['delta']:>+7.2f}% {r['t']:>7.2f} {r['p']:>10.4g} {r['sig']:>4}")

# ------ Agreement check: pooled vs model-level ------
print('\n\nAgreement check (pooled trial-level vs. model-level, α=0.05):')
pooled_by_tier = {r['tier']: r for r in rows}
disagreements = []
for mr in model_level_rows:
    pr = pooled_by_tier[mr['tier']]
    pool_sig = pr['p'] < 0.05
    ml_sig   = mr['p'] < 0.05
    if pool_sig and ml_sig:
        status = 'both sig ✓'
    elif not pool_sig and not ml_sig:
        status = 'both NS'
    else:
        status = f"DISAGREE (pooled p={pr['p']:.3g}, model-level p={mr['p']:.3g})"
        disagreements.append(mr['tier'])
    print(f"  {mr['tier']:<13} {status}")

if not disagreements:
    print('\n✓ All tiers agree. Temperature effect is bulletproof.')
else:
    print(f'\n⚠ Disagreements at: {disagreements}')

Model-level paired t-test (n=16 models per tier):

Tier            t=0.0   t=0.7        Δ       t          p  sig
----------------------------------------------------------
Baseline        0.89%   1.04%   +0.15%    1.41     0.1766     
Job duties      0.83%   1.39%   +0.55%    2.11    0.05045     
Job context     1.06%   1.46%   +0.40%    1.84    0.08325     
Memo            7.20%   7.10%   -0.10%   -0.17     0.8669     
Authority       2.72%   3.13%   +0.40%    2.40    0.02812    *
Policy         14.24%  14.06%   -0.18%   -0.66     0.5201     
Insurance      14.30%  14.21%   -0.08%   -0.22     0.8273     


Agreement check (pooled trial-level vs. model-level, α=0.05):
  Baseline      both NS
  Job duties    DISAGREE (pooled p=1.06e-09, model-level p=0.0505)
  Job context   DISAGREE (pooled p=1.18e-06, model-level p=0.0833)
  Memo          both NS
  Authority     both sig ✓
  Policy        both NS
  Insurance     both NS

⚠ Disagreements at: ['Job duties', 'Job context']
